# 03 \u00b7 Run systems and evaluate

Runs all five systems (A, B, C1, C2, D) on the scripted test debates and scores them, end to end.
It calls `scripts/run_systems.py` then `scripts/evaluate.py` (the same code paths as the CLI), then
renders `outputs/results/metrics.json` as a table.

Needs the trained adapter (`outputs/adapter`) and the passages (symlinked from scratch) \u2014 run on
the cluster. Set `DUMMY = True` in the config cell for a wiring check with toy data and no models.

## Setup

Finds the repo root, puts `src/` on the path, installs missing deps.

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

# Colab check that does NOT raise when the `google` package is absent (it does on the
# SCIS cluster / a local venv). find_spec on a dotted name raises if the parent is missing.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# On Colab, clone the repo once. Elsewhere (cluster / local) it is already present.
if IN_COLAB:
    REPO_URL = "https://github.com/diyadj/argubot.git"
    if not os.path.isdir("argubot"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("argubot")

# Work from the repo root so configs/config.yaml and data/... resolve. Walk up from the launch
# dir (notebooks/ on the cluster), and fall back to ~/argubot (execnotebook runs from your home).
root = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "pyproject.toml").exists()), None)
if root is None and (Path.home() / "argubot" / "pyproject.toml").exists():
    root = Path.home() / "argubot"
if root is not None:
    os.chdir(root)

src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)

pip_name = {"torch": "torch", "transformers": "transformers", "peft": "peft",
            "accelerate": "accelerate", "numpy": "numpy", "yaml": "pyyaml"}
need = [pip_name[m] for m in pip_name if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
if importlib.util.find_spec("fla") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-linear-attention"], check=False)

for _m in [k for k in list(sys.modules) if k == "argubot" or k.startswith("argubot.")]:
    del sys.modules[_m]

import argubot  # noqa: F401
print("cwd:", os.getcwd(), "| IN_COLAB:", IN_COLAB, "| argubot:", argubot.__file__)

## Settings and prerequisite check

Pick the systems and confirm the adapter + passages are present before a real run.

In [ ]:
import os
from argubot.config import load_config

cfg = load_config("configs/config.yaml")

# --- settings ---
DUMMY = False                               # True = toy data + fake components (meaningless numbers)
SYSTEMS = ["A", "B", "C1", "C2", "D"]
BASELINE = "A"
DIALOGUES = "data/processed/test_dialogues.jsonl"   # ignored when DUMMY

# Prerequisite check for a real run: the trained adapter and the passages must be present.
if not DUMMY:
    adapter_ok = os.path.exists(os.path.join(cfg["paths"]["adapter_dir"], "adapter_config.json"))
    passages_ok = os.path.exists(cfg["paths"]["passages"]) or os.path.exists(cfg["paths"]["passages"] + ".gz")
    print("adapter  :", "OK" if adapter_ok else "MISSING", f"({cfg['paths']['adapter_dir']})")
    print("passages :", "OK" if passages_ok else "MISSING", f"({cfg['paths']['passages']})")
    if not adapter_ok:
        print("  -> train it with 02_train.ipynb first (or point paths.adapter_dir at an adapter).")
    if not passages_ok:
        print("  -> link passages from scratch (see docs/DATA_NOTES.md) or run scripts/build_indexes.py.")
else:
    print("DUMMY run: toy dialogues + fake generator/retriever/NLI. Numbers are meaningless.")
print("systems:", SYSTEMS, "| baseline:", BASELINE, "| retrieval.method:", cfg["retrieval"]["method"])

## 1. Run the systems

Writes `outputs/transcripts/<system>.jsonl`. This is the slow step (generation); each file is
saved as its system finishes, so a stopped run keeps the systems already done.

In [ ]:
import subprocess, sys

def run(cmd):
    print("$ " + " ".join(cmd), flush=True)
    if subprocess.run(cmd).returncode != 0:
        raise RuntimeError("command failed: " + " ".join(cmd))

# Generates one transcript per system in outputs/transcripts/<name>.jsonl (persists as it goes).
cmd = [sys.executable, "scripts/run_systems.py", "--systems", *SYSTEMS]
cmd += ["--dummy"] if DUMMY else ["--dialogues", DIALOGUES]
run(cmd)

## 2. Evaluate

Scores contradiction rate (vs the baseline, with bootstrap CIs), distinct-2, claims/turn, and
reply length; writes `outputs/results/metrics.json`.

In [ ]:
# Scores the transcripts, compares each system to the baseline with a paired bootstrap CI,
# writes outputs/results/metrics.json, and prints its own table.
cmd = [sys.executable, "scripts/evaluate.py", "--systems", *SYSTEMS, "--baseline", BASELINE]
cmd += ["--dummy"] if DUMMY else []
run(cmd)

## 3. Results table

The numbers for the report. Lower contradiction rate is better; a 95% CI that excludes 0 means the
difference from the baseline is statistically significant.

In [ ]:
import json
from pathlib import Path

metrics_path = Path(cfg["paths"]["results_dir"]) / "metrics.json"
results = json.loads(metrics_path.read_text())

rows = []
for name in SYSTEMS:
    r = results[name]
    d = r.get("diff_vs_baseline")
    rows.append({
        "System": name,
        "Contra rate": round(r["contradiction_rate"], 3),
        f"Diff vs {BASELINE}": f"{d['mean']:+.3f}" if d else "-",
        "95% CI": f"[{d['ci_low']:+.3f}, {d['ci_high']:+.3f}]" if d else "-",
        "Distinct-2": round(r["distinct_2"], 3),
        "Claims/turn": round(r["claims_per_turn"], 2),
        "Mean words": round(r["mean_reply_words"], 1),
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows).set_index("System")
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string())
except ImportError:
    keys = list(rows[0].keys())
    print(" | ".join(keys))
    for row in rows:
        print(" | ".join(str(row[k]) for k in keys))

print(f"\nSaved to {metrics_path}. A CI that excludes 0 means the system differs significantly from {BASELINE}.")

## For the report

- These are the final numbers on your trained adapter \u2014 record the adapter settings (epochs, data, effective batch) alongside them.
- Check whether B / C1 / C2 / D beat baseline **A** on contradiction rate and whether each 95% CI excludes 0.
- Caveat to state: **D** reranks candidates with the same NLI model family the metric uses, so run the independent (`split_sentences`) contradiction check as a robustness number for D.
- If retrieval evidence looks off-topic (a known weakness in `DATA_NOTES.md`), try `retrieval.method: dense` and re-run.